In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/extracted-josn/latex_extracted.json
/kaggle/input/summary/summarizer2.py


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re
import json
import pandas as pd
from pandas import json_normalize

In [4]:
import torch
print("CUDA available?", torch.cuda.is_available())

CUDA available? True


In [5]:
import sys
sys.path.append('/kaggle/input/summary')

In [6]:
from summarizer2 import Summarizer2

In [7]:
json_path = "/kaggle/input/extracted-josn/latex_extracted.json"
with open(json_path, "r") as f:
    data = json.load(f)

paper = data[0]

In [8]:
model_name = "google-t5/t5-small"   # official repo

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

2025-11-25 05:15:03.669009: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764047703.835769      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764047703.891459      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
## on after checking
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

In [10]:
import os

save_path = "/kaggle/working/t5-small"
os.makedirs(save_path, exist_ok=True)

tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)


In [11]:
def score_equation(eq):
    score = 0

    # Math structure (more structure = important)
    if any(sym in eq for sym in ["=", ":", "+", "-", "*", "^", "_"]):
        score += 2

    # contains mapping
    if "f" in eq or "F" in eq:
        score += 2

    # contains indeterminacy set
    if "I_f" in eq:
        score += 3

    # contains canonical class
    if "K_X" in eq:
        score += 3

    # contains projective space
    if "P^" in eq or "P_" in eq:
        score += 2

    # contains variable structure (likely real math)
    if any(v in eq for v in ["x", "y", "z"]):
        score += 1

    # length reward (meaningful equations, but small weight)
    score += min(len(eq) // 40, 2)

    return score


In [12]:
def get_top5_from_extracted(equations):
    scored = [(eq, score_equation(eq)) for eq in equations]
    scored_sorted = sorted(scored, key=lambda x: x[1], reverse=True)
    return [eq for eq, score in scored_sorted[:5]]

In [13]:
def clean_text(text):
      # 1. Remove LaTeX math blocks
    text = re.sub(r"\$[^$]*\$", " ", text)   # $...$
    text = re.sub(r"\\\[.*?\\\]", " ", text, flags=re.DOTALL)  # \[...\]

    # 2. Remove all LaTeX commands like \alpha, \begin{...}, \mathbb, \frac etc
    text = re.sub(r"\\[a-zA-Z]+(\{[^}]*\})?", " ", text)

    # 3. Remove all {...} braces and content inside
    text = re.sub(r"\{[^{}]*\}", " ", text)

    # 4. Remove all [...]-style tokens (figures, scales, options)
    text = re.sub(r"\[[^\]]*\]", " ", text)

    # 5. Remove standalone parentheses garbage
    text = re.sub(r"\([^)]*\)", " ", text)

    # 6. Remove citation numbers [12], [3], etc
    text = re.sub(r"\[\d+\]", " ", text)

    # 7. Remove any "Figure", "cf.", "see", "Remark", "Proposition", "Lemma"
    text = re.sub(r"\b(Figure|cf|Cf|Remark|Proposition|Lemma)\b[^.]*\.", " ", text)

    # 8. Remove any variable names like X, Y, f, g, π, etc (these confuse T5)
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)

    # 9. Remove repetitive punctuation
    text = re.sub(r"[^\x00-\x7F]+", " ", text)  # remove unicode junk
    text = re.sub(r"[\(\)\[\]\{\}]", " ", text)
    text = re.sub(r"[,;:]+", " ", text)

    # 10. Remove sequences of math-like symbols
    text = re.sub(r"[\|\^\_\=\*\+\-\/]+", " ", text)

    # 11. Remove “. . .”, “...”, repeated dots
    text = re.sub(r"\.{2,}", ".", text)

    # 12. Remove code snippets (Python blocks)
    text = re.sub(r"for .*?:", " ", text)
    text = re.sub(r"return .*", " ", text)

    # 13. Remove anything that looks like file output or variable names
    text = re.sub(r"[A-Za-z0-9_]+\.[A-Za-z0-9_]+", " ", text)  # foo.bar, abc.xyz

    # 14. Remove excess whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [14]:
def clean_paper_for_t5(paper):
    cleaned = {}

    # Clean abstract
    cleaned["abstract"] = clean_text(paper.get("abstract", ""))

    # Clean sections
    cleaned_secs = []
    for sec in paper.get("sections", []):
        cleaned_secs.append({
            "name": clean_text(sec.get("name", "")),
            "content": clean_text(sec.get("content", ""))
        })
    cleaned["sections"] = cleaned_secs

    return cleaned

In [28]:
def t5_generate(prompt, max_len=150):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    out = model.generate(
        inputs["input_ids"],
        num_beams=4,
        length_penalty=1.0,
        no_repeat_ngram_size=2,
        max_length=max_len,
        do_sample=False
    )

    return tokenizer.decode(out[0], skip_special_tokens=True)





In [16]:
def summarize_abstract(paper):
    abstract = paper["abstract"]

    prompt = (
        "Write a clear academic summary of the following abstract. "
        "Begin the first sentence with a phrase like 'This paper investigates', "
        "'This work studies', or 'The authors analyze'. "
        "Explain the main problem, the methods used, the central results, "
        "and why they matter. Do not copy any exact sentences. "
        "Write 5–7 complete scientific sentences.\n\n"
        "Abstract:\n"
        + abstract
        + "\n\nAcademic summary:"
    )

    return t5_generate(prompt)

In [17]:
## remove summary promt after summarise the chunk
def strip_prompt_artifacts(text):
    patterns = [
        r"(?i)Write a clear academic summary of the following abstract.*",
        r"(?i)write\s*5[\–\-]?\s*7.*?sentences[:\.]?",   # "Write 5–7 complete scientific sentences."
        r"(?i)write\s*\d.*?sentences[:\.]?",            
        r"(?i)abstract[:\.]?",
        r"(?i)summarise this chunk.*",
        r"(?i)summarize this chunk.*",
        r"(?i)summarize the following.*",
        r"(?i)Summarize the following text in 2 sentences:*",
        r"(?i)in 2 sentences:*",
        r"(?i)Begin the first sentence with a phrase like*",
        r"(?i)summarize in.*",
        r"(?i)Summary:.*",
        r"(?i)Abstract:*",
        r"(?i)first summary.*",
        r"(?i)second summary.*",
        r"(?i)good academic structure.*",
        r"(?i)abstract:.*",
        r"(?i)this chunk.*",
        r"(?i)this chapter.*",
        r"(?i)write.*academic.*",
        r"(?i)avoid.*",
        r"\.{2,}",        
        r"[\"\']{1,}",    
    ]
    
    cleaned = text
    for p in patterns:
        cleaned = re.sub(p, "", cleaned)
        
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

In [18]:
def build_chunk_prompt(chunk):
    return (
        "Summarize the following text in 2 sentences:"
        + chunk +
        "Summary:"
    )

In [19]:
def split_into_chunks(text, max_words=180):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i+max_words]
        chunks.append(" ".join(chunk_words))
        i += max_words
    return chunks

In [20]:
def summarize_chunk(chunk):
    prompt = build_chunk_prompt(chunk)
    return t5_generate(prompt)

In [21]:
def summarize_section(sec):
    section_name = sec["name"]
    section_text = sec["content"]
   
    summary = summarize_section_t5(section_text)

    return {
        "section_name": section_name,
        "summary": summary
    }

In [22]:
def summarize_section_t5(section_text):

    # Step 2 — split into chunks
    chunks = split_into_chunks(section_text, max_words=180)

    # Step 3 — summarize each chunk
    partial_summaries = []
    for chunk in chunks:
        summary = summarize_chunk(chunk)
        clean_summary = strip_prompt_artifacts(summary)
        partial_summaries.append(clean_summary)
        
    final_summary = " ".join(partial_summaries)
    return final_summary


In [23]:
cleaned_paper = clean_paper_for_t5(paper)
# cleaned_paper

In [24]:
equations = paper.get("equations", [])
important_top5 = get_top5_from_extracted(equations)

In [29]:
abstract_summary = summarize_abstract(cleaned_paper)


In [30]:
abstract_summary

'Write 5–7 complete scientific sentences. Abstract: The present note studies with terms and the indeterminacy locus. We develop an experimental approach based on some Python programming and Machine Learning towards the classification of such maps couple of new explicit is constructed in this way.'

In [31]:
clean_abstruct_summary = strip_prompt_artifacts(abstract_summary)
clean_abstruct_summary

'The present note studies with terms and the indeterminacy locus. We develop an experimental approach based on some Python programming and Machine Learning towards the classification of such maps couple of new explicit is constructed in this way.'

In [32]:
sections = cleaned_paper["sections"]
section_summaries = []
for sec in sections:
    out = summarize_section(sec)
    section_summaries.append(out)
    print("\n=== ", out["section_name"], " ===")
    print(out["summary"])


===  Introduction  ===
Let be complex projective variety and its rational endomorphism. Denote by the indeterminacy locus of. Then is called if the induced morphist is onto. Such maps were introduced and studied from the algebro geometric point of view in the paper. geometry used in this paper can be found in Note that is given by equations. Then the morphism is defined in the following coordinate free way for any point consider the pencil of all linear combinations of which vanish at now identify with the dual plane of lines and put 2 :::::. We develop an experimental approach towards complete description of surjective rational maps which also yields new examples of them. Surjectivity property seems like another substitute for the term generic among many others that one comes across in classical algebraic geometry. For example given smooth conic the set of triples of linear forms satisfying can be compactified into the this is codimension linear section of the Grassmannian and the se

In [34]:
def build_final_json(cleaned_paper, abstract_summary, section_summaries, important_top5, raw_paper):
    # Format sections
    sections_out = []
    for sec in section_summaries:
        sections_out.append({
            "section_name": sec["section_name"],
            "section_summary": sec["summary"].strip()
        })

    # Format important equations
    important_equations = []
    for i, eq in enumerate(important_top5, 1):
        important_equations.append({
            "equation_no": f"Equation {i}",
            "equation": eq
        })

    # Final JSON
    final_json = {
        "title": raw_paper["title"],
        "authors": raw_paper["authors"],
        "abstract_summary": abstract_summary,
        "sections": sections_out,
        "important_equations": important_equations
    }

    return final_json


In [35]:
final_json = build_final_json(
    cleaned_paper=cleaned_paper,
    abstract_summary=clean_abstruct_summary,
    section_summaries=section_summaries,
    important_top5=important_top5,
    raw_paper=paper
)

# Print
import json
print(json.dumps(final_json, indent=4))

{
    "title": "Computations and ML for surjective rational maps",
    "authors": [
        "Ilya Karzhemanov"
    ],
    "abstract_summary": "The present note studies with terms and the indeterminacy locus. We develop an experimental approach based on some Python programming and Machine Learning towards the classification of such maps couple of new explicit is constructed in this way.",
    "sections": [
        {
            "section_name": "Introduction",
            "section_summary": "Let be complex projective variety and its rational endomorphism. Denote by the indeterminacy locus of. Then is called if the induced morphist is onto. Such maps were introduced and studied from the algebro geometric point of view in the paper. geometry used in this paper can be found in Note that is given by equations. Then the morphism is defined in the following coordinate free way for any point consider the pencil of all linear combinations of which vanish at now identify with the dual plane of li

In [ ]:
test_prompt = (
    "Summarize the following text in 2 sentences:\n\n"
    "Machine learning models can detect patterns in data and make predictions. "
    "They are widely used in areas such as healthcare, finance, and robotics. "
    "These models improve when more high-quality data is available.\n\n"
    "Summary:"
)

In [ ]:
output = t5_generate(test_prompt)
print(output)